# LDA Topic Discovery

Reads a CSV file, preprocesses English text, runs Latent Dirichlet Allocation
using collapsed Gibbs sampling, and outputs discovered topics with document assignments.

In [ ]:
import logging
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
import lda

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

## Configuration

In [ ]:
# --- Configuration ---
CSV_FILE = "reddit_cleaned_01_13_first10.csv"
TEXT_COLUMN = "merged_text"
N_TOPICS = 10
N_ITER = 500
N_TOP_WORDS = 10
MAX_FEATURES = None
MIN_DF = 2
MAX_DF = 0.95
ALPHA = 0.1
ETA = 0.01
RANDOM_STATE = 42
OUTPUT_PREFIX = "topic_results"

## 1. Load Data

In [ ]:
df = pd.read_csv(CSV_FILE)
print(f"Columns: {list(df.columns)}")
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Drop rows with missing or empty text
mask = df[TEXT_COLUMN].notna() & (df[TEXT_COLUMN].astype(str).str.strip() != "")
n_dropped = (~mask).sum()
if n_dropped > 0:
    print(f"Dropped {n_dropped} rows with empty text")
df = df[mask].reset_index(drop=True)
documents = df[TEXT_COLUMN].astype(str).tolist()
print(f"Loaded {len(documents)} documents")

## 2. Build Document-Term Matrix

In [ ]:
vectorizer = CountVectorizer(
    max_features=MAX_FEATURES,
    min_df=MIN_DF,
    max_df=MAX_DF,
    stop_words="english",
)
dtm = vectorizer.fit_transform(documents)
vocab = vectorizer.get_feature_names_out()

# Remove all-zero rows
row_sums = np.array(dtm.sum(axis=1)).flatten()
nonzero_mask = row_sums > 0
n_empty = (~nonzero_mask).sum()
if n_empty > 0:
    print(f"{n_empty} documents have no terms after vocabulary filtering")
dtm_filtered = dtm[nonzero_mask]

print(f"Corpus: {dtm_filtered.shape[0]} documents, {dtm_filtered.shape[1]} terms")

## 3. Run LDA

In [ ]:
model = lda.LDA(
    n_topics=N_TOPICS,
    n_iter=N_ITER,
    alpha=ALPHA,
    eta=ETA,
    random_state=RANDOM_STATE,
)
model.fit(dtm_filtered)

## 4. Display Topics

In [ ]:
print(f"{'='*60}")
print(f"Discovered {model.n_topics} Topics")
print(f"{'='*60}\n")

for i, topic_dist in enumerate(model.topic_word_):
    top_indices = np.argsort(topic_dist)[::-1][:N_TOP_WORDS]
    top_words = [(vocab[j], topic_dist[j]) for j in top_indices]
    words_str = ", ".join(f"{w} ({p:.4f})" for w, p in top_words)
    print(f"Topic {i}: {words_str}")

## 5. Assign Topics to Documents

In [ ]:
dominant_topics = np.full(len(df), -1, dtype=int)
topic_probs = np.full(len(df), np.nan)

doc_topic = model.doc_topic_
valid_indices = np.where(nonzero_mask)[0]
for local_idx, global_idx in enumerate(valid_indices):
    dominant_topics[global_idx] = np.argmax(doc_topic[local_idx])
    topic_probs[global_idx] = doc_topic[local_idx].max()

df["dominant_topic"] = dominant_topics
df["topic_probability"] = topic_probs

df[[TEXT_COLUMN, "dominant_topic", "topic_probability"]].head(10)

## 6. Save Results

In [ ]:
# Save document assignments
assignments_path = f"{OUTPUT_PREFIX}_assignments.csv"
df.to_csv(assignments_path, index=False)
print(f"Document assignments saved to: {assignments_path}")

# Save topic descriptions
topic_rows = []
for i, topic_dist in enumerate(model.topic_word_):
    top_indices = np.argsort(topic_dist)[::-1][:N_TOP_WORDS]
    row = {"topic_id": i}
    for rank, j in enumerate(top_indices):
        row[f"word_{rank+1}"] = vocab[j]
        row[f"prob_{rank+1}"] = round(float(topic_dist[j]), 6)
    topic_rows.append(row)

topics_path = f"{OUTPUT_PREFIX}_topics.csv"
pd.DataFrame(topic_rows).to_csv(topics_path, index=False)
print(f"Topic descriptions saved to: {topics_path}")
print("\nDone!")